# Inference Demo

Production-ready demonstration notebook for the Causal Multi-Modal DTI Framework.

This notebook shows how to load a trained model, perform inference on a SMILES string and protein sequence, visualize attention/attributions, and export predictions.

## 1. Imports

In [ ]:
from pathlib import Path
import torch
import pandas as pd
import numpy as np

from models.full_model import build_model
from datasets.preprocess import smiles_to_graph
from datasets.protein_dataset import ProteinRepository, ProteinEmbeddingCache


## 2. Repository Setup

In [ ]:
ROOT = Path('..').resolve()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(ROOT)
print('Device:', DEVICE)


## 3. Load Configuration

In [ ]:
import yaml
cfg = yaml.safe_load(open(ROOT/'configs'/'bindingdb.yaml'))
cfg


## 4. Load Model

In [ ]:
repo = ProteinRepository.load(ROOT/'datasets'/'processed'/'protein_repository.pkl')
cache = ProteinEmbeddingCache.load(ROOT/'datasets'/'processed'/'esm_embeddings.pt')

model = build_model(cfg, protein_repository=repo, embedding_cache=cache)
ckpt = torch.load(ROOT/'checkpoints'/'bindingdb'/'best_model.pth', map_location=DEVICE)
state = ckpt.get('model_state', ckpt.get('model_state_dict'))
model.load_state_dict(state)
model.to(DEVICE)
model.eval()
print('Model loaded.')


## 5. Example Input

In [ ]:
smiles = 'CC(=O)OC1=CC=CC=C1C(=O)O'   # Aspirin
protein_sequence = (
    'MTEITAAMVKELRESTGAGMMDCKNALSETQHEK'
)
print(smiles)
print(protein_sequence)


## 6. Build Input

In [ ]:
graph = smiles_to_graph(smiles)
protein_id = repo.add_sequence(protein_sequence) if hasattr(repo,'add_sequence') else 0


## 7. Run Inference

In [ ]:
with torch.no_grad():
    output = model(graph, [protein_id])

print(output)

prediction = output.get('prediction')
uncertainty = output.get('uncertainty', None)

print('Prediction:', prediction)
print('Uncertainty:', uncertainty)


## 8. Attention Visualization

In [ ]:
import matplotlib.pyplot as plt

attn = output.get('cross_attention', None)
if attn is not None:
    A = attn.mean(0).cpu().numpy()
    plt.figure(figsize=(8,6))
    plt.imshow(A, aspect='auto')
    plt.colorbar()
    plt.title('Cross-Modal Attention')
    plt.xlabel('Protein Residues')
    plt.ylabel('Drug Atoms')
    plt.show()
else:
    print('Attention not returned by model.')


## 9. Attribution Visualization

In [ ]:
drug_attr = output.get('drug_attribution', None)
prot_attr = output.get('protein_attribution', None)

if drug_attr is not None:
    plt.figure(figsize=(10,3))
    plt.bar(range(len(drug_attr)), drug_attr.cpu().numpy())
    plt.title('Drug Atom Attribution')
    plt.show()

if prot_attr is not None:
    plt.figure(figsize=(10,3))
    plt.plot(prot_attr.cpu().numpy())
    plt.title('Protein Residue Attribution')
    plt.show()


## 10. Batch Prediction

In [ ]:
pairs = pd.DataFrame({
    'smiles':[smiles],
    'sequence':[protein_sequence]
})

results = []

for _, row in pairs.iterrows():
    # Replace with repository batch pipeline
    results.append({
        'smiles': row.smiles,
        'prediction': float(prediction.cpu().item()) if prediction is not None else np.nan
    })

results = pd.DataFrame(results)
results


## 11. Export Results

In [ ]:
out = ROOT/'outputs'/'demo_predictions.csv'
out.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(out, index=False)
print(out)


## 12. Next Steps

- Run docking validation
- Generate explainability figures
- Export calibration curves
- Evaluate on BindingDB/DAVIS/KIBA test sets